# RT-DETR-X 2-class training (~10h budget)

Notebook nay chay cho Kaggle `Run All`.

- Dau vao: dataset detect 5 class (YOLO format).
- Chuyen doi: 2 class detect (`crack` / `no_crack`).
- Model: `rtdetr-x.pt`.
- Tu canh epoch dua tren 1 lan calibration de tong thoi gian gan `TARGET_HOURS`.
- Xuat bundle de ban co the train tiep o notebook thu 2.


In [ ]:
!pip -q install -U ultralytics pyyaml tqdm

import csv
import gc
import json
import os
import random
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import torch
import yaml
from tqdm.auto import tqdm
from ultralytics import RTDETR

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), '| GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU{i}:', torch.cuda.get_device_name(i))


In [ ]:
# ===== User config =====
DATASET_ROOT_HINT = None  # Vi du: '/kaggle/input/ten-dataset' (de None de auto-find)
DATASET_HINT_KEYWORD = 'crack'

MODEL_NAME = 'rtdetr-x.pt'
RUN_NAME_BASE = 'rtdetr_x_crack_full_stable_v1'

TRAIN_MODE = 'full'  # 'full' or 'time_budget'
FULL_PHASE1_EPOCHS = 15
FULL_PHASE2_EPOCHS = 0

TARGET_HOURS = 10.0  # chi dung khi TRAIN_MODE = 'time_budget'
IMAGE_SIZE = 1024
USE_AMP = False  # RT-DETR tren T4: tat AMP de giam nguy co NaN
RUN_CALIBRATION = True
CALIBRATION_EPOCHS = 1
CALIBRATION_OVERHEAD_MIN = 12.0

MIN_TOTAL_EPOCHS = 10
MAX_TOTAL_EPOCHS = 40
PHASE2_RATIO = 0.20
MIN_PHASE2_EPOCHS = 3

WORK_ROOT = Path('/kaggle/working/input_dataset_crack_rtdetr')
RUNTIME_YAML = Path('/kaggle/working/data_runtime_crack_rtdetr.yaml')
PROJECT_DIR = Path('/kaggle/working/runs/detect')
REPORT_JSON = Path('/kaggle/working/report_metrics_rtdetr_crack.json')
REPORT_CSV = Path('/kaggle/working/report_metrics_rtdetr_crack.csv')

TRAIN_DEVICE = '0,1' if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else ('0' if torch.cuda.is_available() else 'cpu')
EVAL_DEVICE = 0 if torch.cuda.is_available() else 'cpu'
BATCH_CANDIDATES = [4, 2, 1] if torch.cuda.device_count() >= 2 else ([2, 1] if torch.cuda.is_available() else [1])
EVAL_BATCH = 1

print('Train device      :', TRAIN_DEVICE)
print('Batch candidates  :', BATCH_CANDIDATES)
print('Time target (hour):', TARGET_HOURS)


In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def read_yaml(path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

def find_data_yaml(dataset_root_hint=None, keyword='crack'):
    if dataset_root_hint:
        p = Path(dataset_root_hint)
        if p.is_file() and p.name == 'data.yaml':
            return p.resolve()
        if p.is_dir() and (p / 'data.yaml').exists():
            return (p / 'data.yaml').resolve()

    cands = []
    for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not base.exists():
            continue
        for p in base.rglob('data.yaml'):
            s = p.as_posix().lower()
            score = 0
            if keyword and keyword.lower() in s:
                score += 10
            if 'train' in s:
                score += 1
            cands.append((score, len(s), p.resolve()))
    if not cands:
        return None
    cands.sort(key=lambda x: (-x[0], x[1]))
    return cands[0][2]

def infer_dataset_root_from_dirs(keyword='crack'):
    cands = []
    for base in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not base.exists():
            continue
        for train_images in base.rglob('train/images'):
            root = train_images.parent.parent
            val_name = None
            if (root / 'val' / 'images').exists():
                val_name = 'val'
            elif (root / 'valid' / 'images').exists():
                val_name = 'valid'
            if val_name is None:
                continue
            score = 0
            s = root.as_posix().lower()
            if keyword and keyword.lower() in s:
                score += 10
            if (root / 'test' / 'images').exists():
                score += 2
            if (root / 'train' / 'labels').exists() and (root / val_name / 'labels').exists():
                score += 3
            cands.append((score, len(s), root.resolve(), val_name))
    if not cands:
        raise FileNotFoundError('Khong tim thay data.yaml va cung khong suy luan duoc split train/val tu thu muc du lieu')
    cands.sort(key=lambda x: (-x[0], x[1]))
    _, _, root, val_name = cands[0]
    return root, val_name

def resolve_split_path(data_yaml_path, cfg, split_key):
    raw = cfg.get(split_key)
    if raw is None:
        return None
    p = Path(str(raw))
    if p.is_absolute():
        return p
    root = cfg.get('path', None)
    if root:
        root_p = Path(str(root))
        if not root_p.is_absolute():
            root_p = (data_yaml_path.parent / root_p).resolve()
        return (root_p / p).resolve()
    return (data_yaml_path.parent / p).resolve()

def ensure_clean_dir(p):
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True, exist_ok=True)

def list_images(image_dir):
    if image_dir is None or (not image_dir.exists()):
        return []
    return sorted([x for x in image_dir.rglob('*') if x.is_file() and x.suffix.lower() in IMG_EXTS])

def yolo_seg_to_bbox(coords):
    xs = coords[0::2]
    ys = coords[1::2]
    x1, x2 = max(0.0, min(xs)), min(1.0, max(xs))
    y1, y2 = max(0.0, min(ys)), min(1.0, max(ys))
    w = max(1e-9, x2 - x1)
    h = max(1e-9, y2 - y1)
    return [x1 + w / 2.0, y1 + h / 2.0, w, h]

def normalize_bbox(b):
    cx, cy, w, h = [float(v) for v in b]
    cx = min(1.0, max(0.0, cx))
    cy = min(1.0, max(0.0, cy))
    w = min(1.0, max(1e-9, w))
    h = min(1.0, max(1e-9, h))
    return [cx, cy, w, h]

def parse_label_as_crack_boxes(label_path):
    crack_boxes = []
    invalid = 0
    if not label_path.exists():
        return crack_boxes, invalid
    for raw in label_path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5:
            invalid += 1
            continue
        try:
            vals = [float(x) for x in parts[1:]]
        except Exception:
            invalid += 1
            continue
        if len(parts) == 5:
            box = normalize_bbox(vals[:4])
            crack_boxes.append(box)
        elif len(vals) >= 6 and len(vals) % 2 == 0:
            box = normalize_bbox(yolo_seg_to_bbox(vals))
            crack_boxes.append(box)
        else:
            invalid += 1
    return crack_boxes, invalid

def link_or_copy(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
        return
    except Exception:
        pass
    try:
        os.symlink(src, dst)
        return
    except Exception:
        pass
    shutil.copy2(src, dst)

data_yaml = find_data_yaml(DATASET_ROOT_HINT, DATASET_HINT_KEYWORD)
if data_yaml is not None:
    cfg = read_yaml(data_yaml)
    val_key = 'val' if 'val' in cfg else ('valid' if 'valid' in cfg else None)
    if val_key is None:
        raise SystemExit('data.yaml khong co val/valid')
    discover_mode = 'from_data_yaml'
else:
    inferred_root, val_key = infer_dataset_root_from_dirs(DATASET_HINT_KEYWORD)
    cfg = {
        'path': str(inferred_root),
        'train': 'train/images',
        val_key: f'{val_key}/images',
    }
    if (inferred_root / 'test' / 'images').exists():
        cfg['test'] = 'test/images'
    data_yaml = inferred_root / 'data.yaml'
    discover_mode = 'from_split_dirs'

split_src = {
    'train': resolve_split_path(data_yaml, cfg, 'train'),
    'val': resolve_split_path(data_yaml, cfg, val_key),
    'test': resolve_split_path(data_yaml, cfg, 'test') if 'test' in cfg else None,
}

for req in ['train', 'val']:
    p = split_src.get(req)
    if p is None or (not p.exists()):
        raise SystemExit(f'Split {req} khong ton tai: {p}')
    n_img = sum(1 for x in p.rglob('*') if x.is_file() and x.suffix.lower() in IMG_EXTS)
    if n_img == 0:
        raise SystemExit(f'Split {req} khong co image: {p}')

ensure_clean_dir(WORK_ROOT)
stats = {}

for split_name in ['train', 'val', 'test']:
    src_img_dir = split_src.get(split_name)
    if src_img_dir is None or (not src_img_dir.exists()):
        continue
    src_label_dir = src_img_dir.parent / 'labels'
    out_img_dir = WORK_ROOT / split_name / 'images'
    out_lbl_dir = WORK_ROOT / split_name / 'labels'
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    images = list_images(src_img_dir)
    crack_images = 0
    no_crack_images = 0
    crack_boxes = 0
    invalid_lines = 0

    for img_path in tqdm(images, desc=f'convert {split_name}'):
        rel = img_path.relative_to(src_img_dir)
        dst_img = out_img_dir / rel
        link_or_copy(img_path, dst_img)

        src_lbl = src_label_dir / rel.with_suffix('.txt')
        dst_lbl = out_lbl_dir / rel.with_suffix('.txt')
        dst_lbl.parent.mkdir(parents=True, exist_ok=True)

        boxes, bad = parse_label_as_crack_boxes(src_lbl)
        invalid_lines += bad
        if len(boxes) > 0:
            crack_images += 1
            crack_boxes += len(boxes)
            lines = [f"0 {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}" for b in boxes]
            dst_lbl.write_text('\n'.join(lines) + '\n', encoding='utf-8')
        else:
            no_crack_images += 1
            # Negative image in detect task must have an empty label file.
            dst_lbl.write_text('', encoding='utf-8')

    stats[split_name] = {
        'images': len(images),
        'crack_images': crack_images,
        'no_crack_images': no_crack_images,
        'crack_boxes': crack_boxes,
        'invalid_label_lines': invalid_lines,
    }

runtime = {
    'path': str(WORK_ROOT),
    'train': 'train/images',
    'val': 'val/images',
    'nc': 1,
    'names': {0: 'crack'},
}
if (WORK_ROOT / 'test' / 'images').exists():
    runtime['test'] = 'test/images'

RUNTIME_YAML.write_text(yaml.safe_dump(runtime, sort_keys=False, allow_unicode=False), encoding='utf-8')

print('dataset detect  :', discover_mode)
print('data.yaml source :', data_yaml)
print('runtime yaml     :', RUNTIME_YAML)
print(json.dumps(stats, indent=2))

if stats.get('train', {}).get('images', 0) == 0:
    raise SystemExit('Train split rong sau convert')
if stats.get('val', {}).get('images', 0) == 0:
    raise SystemExit('Val split rong sau convert')
if stats.get('train', {}).get('crack_images', 0) == 0:
    raise SystemExit('Train khong co anh crack')
if stats.get('train', {}).get('no_crack_images', 0) == 0:
    print('WARNING: Train split khong co negative image (no_crack).')

print('PREFLIGHT PASS')


In [ ]:
def empty_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def train_with_batch_fallback(model_source, run_name, train_args, batch_candidates):
    last_error = None
    for b in batch_candidates:
        empty_cache()
        t0 = time.time()
        print(f'\nStart {run_name} | model={model_source} | batch={b}')
        try:
            model = RTDETR(model_source)
            model.train(name=run_name, batch=int(b), **train_args)
            elapsed_min = (time.time() - t0) / 60.0
            save_dir = Path(train_args['project']) / run_name
            return {
                'ok': True,
                'batch': int(b),
                'elapsed_min': float(elapsed_min),
                'save_dir': str(save_dir),
                'error': None,
            }
        except Exception as e:
            last_error = e
            msg = str(e).lower()
            print(f'[train] batch={b} failed: {e}')
            if ('out of memory' in msg) or ('cuda error' in msg) or ('cudnn' in msg):
                continue
            raise
    raise RuntimeError(f'All batch attempts failed. Last error: {last_error}')

common_train_args = dict(
    data=str(RUNTIME_YAML),
    device=TRAIN_DEVICE,
    imgsz=IMAGE_SIZE,
    optimizer='AdamW',
    weight_decay=0.0005,
    warmup_epochs=3.0,
    cos_lr=True,
    pretrained=True,
    amp=USE_AMP,
    cache=False,
    workers=4,
    save=True,
    save_period=10,
    plots=True,
    val=True,
    verbose=True,
    project=str(PROJECT_DIR),
    exist_ok=True,
    seed=SEED,
    deterministic=False,
)

calib_minutes_per_epoch = None
calib_info = None


def check_results_health(run_dir):
    csv_path = Path(run_dir) / 'results.csv'
    if not csv_path.exists():
        return
    with csv_path.open('r', encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    if not rows:
        return

    bad = []
    for i, row in enumerate(rows, start=1):
        for k, v in row.items():
            if 'loss' not in k.lower():
                continue
            try:
                fv = float(v)
            except Exception:
                continue
            if (not np.isfinite(fv)):
                bad.append((i, k, v))
                if len(bad) >= 5:
                    break
        if len(bad) >= 5:
            break
    if bad:
        sample = '; '.join([f'epoch_row={r} {k}={v}' for r, k, v in bad])
        raise RuntimeError(f'Run co NaN/Inf trong results.csv: {sample}')


if str(TRAIN_MODE).lower() == 'full':
    phase1_epochs = int(FULL_PHASE1_EPOCHS)
    phase2_epochs = int(FULL_PHASE2_EPOCHS)
    est_total_epochs = phase1_epochs + phase2_epochs
    calib_minutes_per_epoch = None
    print('Mode                 : FULL')
    print('Phase1 epochs        :', phase1_epochs)
    print('Phase2 epochs        :', phase2_epochs)
    print('Total epochs         :', est_total_epochs)
else:
    if RUN_CALIBRATION:
        calib_args = dict(common_train_args)
        calib_args.update(dict(
            epochs=int(CALIBRATION_EPOCHS),
            lr0=0.0002,
            lrf=0.01,
            patience=0,
            hsv_h=0.015,
            hsv_s=0.7,
            hsv_v=0.4,
            degrees=1.5,
            translate=0.06,
            scale=0.30,
            fliplr=0.5,
            flipud=0.0,
            mosaic=0.20,
            mixup=0.03,
            close_mosaic=12,
            erasing=0.15,
        ))
        calib_name = f'{RUN_NAME_BASE}_calib'
        calib_info = train_with_batch_fallback(MODEL_NAME, calib_name, calib_args, BATCH_CANDIDATES)
        calib_minutes_per_epoch = calib_info['elapsed_min'] / max(1, int(CALIBRATION_EPOCHS))
    else:
        # Conservative default for RT-DETR-X 1024 on T4 class GPU.
        calib_minutes_per_epoch = 17.0 if torch.cuda.is_available() else 60.0

    target_minutes = float(TARGET_HOURS) * 60.0
    usable_minutes = max(60.0, target_minutes - float(CALIBRATION_OVERHEAD_MIN) - (calib_info['elapsed_min'] if calib_info else 0.0))
    est_total_epochs = int(round(usable_minutes / max(1e-6, calib_minutes_per_epoch)))
    est_total_epochs = max(int(MIN_TOTAL_EPOCHS), min(int(MAX_TOTAL_EPOCHS), est_total_epochs))

    phase2_epochs = max(int(MIN_PHASE2_EPOCHS), int(round(est_total_epochs * float(PHASE2_RATIO))))
    phase1_epochs = max(10, est_total_epochs - phase2_epochs)

    print('Mode                 : TIME_BUDGET')
    print('Calibration min/epoch:', round(calib_minutes_per_epoch, 3))
    print('Estimated total      :', est_total_epochs)
    print('Phase1 epochs        :', phase1_epochs)
    print('Phase2 epochs        :', phase2_epochs)


In [ ]:
phase1_args = dict(common_train_args)
phase1_args.update(dict(
    epochs=int(phase1_epochs),
    lr0=0.0002,
    lrf=0.01,
    patience=20,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=1.5,
    translate=0.06,
    scale=0.30,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.20,
    mixup=0.03,
    close_mosaic=12,
    erasing=0.15,
))

phase1_name = f'{RUN_NAME_BASE}_phase1'
phase1 = train_with_batch_fallback(MODEL_NAME, phase1_name, phase1_args, BATCH_CANDIDATES)
phase1_dir = Path(phase1['save_dir'])
check_results_health(phase1_dir)
phase1_best = phase1_dir / 'weights' / 'best.pt'
phase1_last = phase1_dir / 'weights' / 'last.pt'
assert phase1_best.exists(), f'Missing phase1 best: {phase1_best}'

phase2 = None
phase2_dir = None
final_best = phase1_best
final_last = phase1_last

if int(phase2_epochs) > 0:
    phase2_args = dict(common_train_args)
    phase2_args.update(dict(
        epochs=int(phase2_epochs),
        lr0=0.00005,
        lrf=0.05,
        patience=8,
        hsv_h=0.010,
        hsv_s=0.35,
        hsv_v=0.2,
        degrees=0.5,
        translate=0.02,
        scale=0.12,
        fliplr=0.5,
        flipud=0.0,
        mosaic=0.0,
        mixup=0.0,
        close_mosaic=1,
        erasing=0.0,
        save_period=0,
    ))

    phase2_name = f'{RUN_NAME_BASE}_phase2'
    phase2 = train_with_batch_fallback(str(phase1_best), phase2_name, phase2_args, BATCH_CANDIDATES)
    phase2_dir = Path(phase2['save_dir'])
    check_results_health(phase2_dir)
    final_best = phase2_dir / 'weights' / 'best.pt'
    final_last = phase2_dir / 'weights' / 'last.pt'
    assert final_best.exists(), f'Missing final best: {final_best}'
else:
    print('Skip phase2 vi phase2_epochs =', phase2_epochs)

print('phase1:', phase1)
print('phase2:', phase2)
print('final_best:', final_best)


In [ ]:
def pack_metrics(metrics):
    return {
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
    }

empty_cache()
model = RTDETR(str(final_best))
val_m = model.val(data=str(RUNTIME_YAML), split='val', device=EVAL_DEVICE, imgsz=IMAGE_SIZE, batch=EVAL_BATCH, verbose=True)

runtime_loaded = read_yaml(RUNTIME_YAML)
test_pack = None
if 'test' in runtime_loaded:
    test_m = model.val(data=str(RUNTIME_YAML), split='test', device=EVAL_DEVICE, imgsz=IMAGE_SIZE, batch=EVAL_BATCH, verbose=True)
    test_pack = pack_metrics(test_m)

report = {
    'model_name': MODEL_NAME,
    'run_name_base': RUN_NAME_BASE,
    'target_hours': TARGET_HOURS,
    'image_size': IMAGE_SIZE,
    'train_device': TRAIN_DEVICE,
    'batch_candidates': BATCH_CANDIDATES,
    'calibration': calib_info,
    'calib_minutes_per_epoch': calib_minutes_per_epoch,
    'phase1_epochs': phase1_epochs,
    'phase2_epochs': phase2_epochs,
    'phase1': phase1,
    'phase2': phase2,
    'final_best': str(final_best),
    'val': pack_metrics(val_m),
    'test': test_pack,
    'dataset_stats': stats,
}

REPORT_JSON.write_text(json.dumps(report, indent=2), encoding='utf-8')
with REPORT_CSV.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['split', 'mAP50', 'mAP50-95', 'precision', 'recall'])
    writer.writeheader()
    writer.writerow({'split': 'val', **report['val']})
    if report['test']:
        writer.writerow({'split': 'test', **report['test']})

print(json.dumps(report, indent=2))
print('Saved:', REPORT_JSON)


In [ ]:
bundle = Path('/kaggle/working') / f'{RUN_NAME_BASE}_bundle.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [RUNTIME_YAML, REPORT_JSON, REPORT_CSV]:
        if p.exists():
            zf.write(p, arcname=p.name)
    for p in [phase1_best, phase1_last, final_best, final_last]:
        if p.exists():
            zf.write(p, arcname=f'weights/{p.name}')
    run_dirs = [(phase1_dir, 'phase1')]
    if phase2_dir is not None:
        run_dirs.append((phase2_dir, 'phase2'))
    for run_dir, tag in run_dirs:
        for rel in ['args.yaml', 'results.csv', 'results.png', 'PR_curve.png', 'P_curve.png', 'R_curve.png', 'F1_curve.png']:
            p = run_dir / rel
            if p.exists():
                zf.write(p, arcname=f'{tag}/{rel}')

print('Bundle saved:', bundle)
print('Size MB:', round(bundle.stat().st_size / (1024 * 1024), 2))
